# 02. 데이터 품질 + 텍스트 분석

**목표**: 데이터 품질 이슈 발견 + 텍스트 길이 분포로 청킹 전략 도출

**의존**: `eda_output/phase1_inventory.json`, `eda_output/phase2_schema.json`

**산출물**: `eda_output/phase3_quality.json`, `eda_output/phase4_text.json`

In [1]:
# ── 환경 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import re
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.auto import tqdm

from scripts.eda.common import (
    DATA_DIR,
    get_sample,
    load_all,
    load_result,
    save_result,
)
from scripts.eda.data_registry import CATEGORIES

pio.templates.default = "plotly_white"

# ┌──────────────────────────────────────────────────────────┐
# │  분석 모드 선택                                          │
# │                                                          │
# │  True  = 전체 데이터 로드 (파일의 모든 레코드)           │
# │          대용량 파일은 시간 + 메모리 소요                │
# │                                                          │
# │  False = 샘플 데이터 (카테고리당 5,000건)                │
# │          빠른 미리보기용                                  │
# └──────────────────────────────────────────────────────────┘
USE_FULL_DATA = True

# 이전 단계 결과 로드
phase1 = load_result("phase1_inventory")
phase2 = load_result("phase2_schema")
print(f"Phase 1 로드: {len(phase1)}개 파일")
print(f"Phase 2 로드: {len(phase2)}개 카테고리 스키마")

# ── 데이터 로딩 ────────────────────────────────────────────
SAMPLE_SIZE = 5000
_data_cache: dict[str, list[dict]] = {}

mode = "전체 데이터 (모든 레코드)" if USE_FULL_DATA else f"샘플 데이터 ({SAMPLE_SIZE:,}건)"
print(f"\n분석 모드: {mode}")

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="데이터 로딩"):
    first_file = cat_info["files"][0]
    filepath = DATA_DIR / first_file
    if not filepath.exists():
        continue
    if USE_FULL_DATA:
        _data_cache[cat_key] = load_all(filepath)
    else:
        _data_cache[cat_key] = get_sample(filepath, n=SAMPLE_SIZE, fast=True)
    print(f"  {cat_info['label']}: {len(_data_cache[cat_key]):,}건")

print(f"\n총 {len(_data_cache)}개 카테고리 로드 완료")

c:\Users\fkjy1\dev\boot_camp\law-3\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Phase 1 로드: 48개 파일
Phase 2 로드: 11개 카테고리 스키마

분석 모드: 전체 데이터 (모든 레코드)


데이터 로딩:   9%|▉         | 1/11 [00:06<01:02,  6.23s/it]

  판례: 92,055건


데이터 로딩:  18%|█▊        | 2/11 [00:08<00:36,  4.00s/it]

...

  법률용어사전: 81,488건


데이터 로딩:  91%|█████████ | 10/11 [00:13<00:00,  1.90it/s]

  조약: 3,589건


데이터 로딩: 100%|██████████| 11/11 [00:13<00:00,  1.26s/it]

  행정규칙: 5,258건

총 11개 카테고리 로드 완료


## 1. 데이터 품질 분석

카테고리별 5,000건 샘플에서 null/empty 비율, 중복 ID, 인코딩 이슈를 점검합니다.

In [2]:
# ── 카테고리별 품질 분석 ──────────────────────────────────
quality_results = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="품질 분석"):
    if cat_key not in _data_cache:
        continue

    sample = _data_cache[cat_key]
    total = len(sample)
    if total == 0:
        continue

    # 필드별 null/empty 분석
    field_quality: dict[str, dict] = {}
    for record in sample:
        for key, value in record.items():
            if key not in field_quality:
                field_quality[key] = {"null": 0, "empty": 0, "present": 0, "null_str": 0}
            fq = field_quality[key]
            fq["present"] += 1
            if value is None:
                fq["null"] += 1
            elif isinstance(value, str):
                if value.strip() == "":
                    fq["empty"] += 1
                elif value.strip().lower() == "null":
                    fq["null_str"] += 1

    # ID 중복 분석
    id_field = cat_info.get("id_field")
    duplicate_ids = 0
    duplicate_rate = 0.0
    if id_field:
        # ID 값이 list인 경우 str로 변환하여 hashable하게 처리
        ids = []
        for r in sample:
            v = r.get(id_field)
            if v is None:
                continue
            ids.append(str(v) if isinstance(v, list) else v)
        id_counts = Counter(ids)
        duplicate_ids = sum(1 for c in id_counts.values() if c > 1)
        duplicate_rate = duplicate_ids / len(id_counts) if id_counts else 0

    # 인코딩 이슈 탐지
    encoding_issues = 0
    html_entities = 0
    for record in sample[:1000]:  # 1000건만 체크
        for value in record.values():
            if not isinstance(value, str):
                continue
            if "\ufffd" in value or "\\u" in value:
                encoding_issues += 1
            if "&amp;" in value or "&lt;" in value or "&gt;" in value:
                html_entities += 1

    quality_results[cat_key] = {
        "label": cat_info["label"],
        "sample_count": total,
        "field_quality": {
            k: {
                "null_rate": round(v["null"] / v["present"], 4) if v["present"] > 0 else 0,
                "empty_rate": round(v["empty"] / v["present"], 4) if v["present"] > 0 else 0,
                "null_str_count": v["null_str"],
            }
            for k, v in field_quality.items()
        },
        "duplicate_id_field": id_field,
        "duplicate_ids": duplicate_ids,
        "duplicate_rate": round(duplicate_rate, 4),
        "encoding_issues": encoding_issues,
        "html_entities": html_entities,
    }
    print(f"  {cat_info['label']}: dup={duplicate_rate:.2%}, encoding={encoding_issues}, html={html_entities}")

품질 분석:   9%|▉         | 1/11 [00:02<00:24,  2.43s/it]

  판례: dup=0.00%, encoding=0, html=0
  법령: dup=0.00%, encoding=0, html=0


품질 분석:  27%|██▋       | 3/11 [00:03<00:06,  1.14it/s]

  헌재결정례: dup=0.00%, encoding=0, html=3


품질 분석:  36%|███▋      | 4/11 [00:03<00:05,  1.18it/s]

  행정심판례: dup=0.00%, encoding=0, html=4


품질 분석:  55%|█████▍    | 6/11 [00:04<00:02,  1.97it/s]

  특별행정심판: dup=0.00%, encoding=0, html=0
  법령해석례: dup=0.00%, encoding=0, html=0
  위원회 결정문: dup=0.00%, encoding=0, html=0
  부처 해석례: dup=0.00%, encoding=0, html=0


품질 분석: 100%|██████████| 11/11 [00:04<00:00,  2.31it/s]

  법률용어사전: dup=2.21%, encoding=0, html=1
  조약: dup=0.00%, encoding=0, html=6
  행정규칙: dup=0.02%, encoding=0, html=0


In [3]:
# ── null률 히트맵 (카테고리 x 필드) ──────────────────────
# 주요 필드: text_fields + id_field + date_field + summary_field
key_fields_per_cat = {}
for cat_key, cat_info in CATEGORIES.items():
    fields = set()
    if cat_info.get("id_field"):
        fields.add(cat_info["id_field"])
    if cat_info.get("date_field"):
        fields.add(cat_info["date_field"])
    for tf in cat_info.get("text_fields", []):
        fields.add(tf)
    if cat_info.get("summary_field"):
        fields.add(cat_info["summary_field"])
    key_fields_per_cat[cat_key] = fields

# 모든 주요 필드의 합집합
all_key_fields = sorted(set().union(*key_fields_per_cat.values()))

# 히트맵 데이터 구성
heatmap_data = []
cat_labels = []
for cat_key, qr in quality_results.items():
    cat_labels.append(qr["label"])
    row = []
    for field in all_key_fields:
        fq = qr["field_quality"].get(field)
        if fq is None:
            row.append(None)  # 해당 카테고리에 이 필드 없음
        else:
            row.append(fq["null_rate"] + fq["empty_rate"])  # null + empty 합산
    heatmap_data.append(row)

fig = px.imshow(
    heatmap_data,
    x=all_key_fields,
    y=cat_labels,
    title="카테고리 x 주요 필드 null/empty 비율 히트맵<br><sub>text_fields + summary_field + id/date 필드 포함</sub>",
    labels=dict(x="필드", y="카테고리", color="null+empty 비율"),
    color_continuous_scale="YlOrRd",
    aspect="auto",
)
fig.update_layout(height=500)
fig.show()

In [4]:
# ── 중복 ID 비율 바 차트 ─────────────────────────────────
dup_data = [
    {
        "카테고리": qr["label"],
        "중복 비율": qr["duplicate_rate"],
        "중복 ID 수": qr["duplicate_ids"],
        "ID 필드": qr["duplicate_id_field"] or "없음",
    }
    for qr in quality_results.values()
]
df_dup = pd.DataFrame(dup_data).sort_values("중복 비율", ascending=True)

fig = px.bar(
    df_dup,
    x="중복 비율",
    y="카테고리",
    orientation="h",
    title="카테고리별 ID 중복 비율",
    hover_data=["중복 ID 수", "ID 필드"],
    color="중복 비율",
    color_continuous_scale="Reds",
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 2. 텍스트 길이 분석

주요 텍스트 필드의 길이 분포를 분석하여 청킹 전략을 도출합니다.

In [5]:
# ── 카테고리별 텍스트 길이 수집 (원본 + 요약 필드) ─────────
text_stats = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="텍스트 분석"):
    text_fields = list(cat_info.get("text_fields", []))
    summary_field = cat_info.get("summary_field")

    # 요약 필드가 있으면 분석 대상에 추가
    if summary_field:
        text_fields.append(summary_field)

    if not text_fields:
        continue

    if cat_key not in _data_cache:
        continue

    data = _data_cache[cat_key]

    field_lengths: dict[str, list[int]] = {f: [] for f in text_fields}

    for record in data:
        for field in text_fields:
            value = record.get(field)
            if isinstance(value, str) and value.strip():
                field_lengths[field].append(len(value))
            elif isinstance(value, list):
                # 배열 필드 (조문내용 등): 전체 직렬화 길이
                text = str(value)
                field_lengths[field].append(len(text))

    cat_stats = {}
    for field, lengths in field_lengths.items():
        if not lengths:
            continue
        arr = np.array(lengths)
        is_summary = (field == summary_field)
        cat_stats[field] = {
            "count": len(lengths),
            "min": int(arr.min()),
            "p25": int(np.percentile(arr, 25)),
            "p50": int(np.percentile(arr, 50)),
            "p75": int(np.percentile(arr, 75)),
            "p90": int(np.percentile(arr, 90)),
            "p99": int(np.percentile(arr, 99)),
            "max": int(arr.max()),
            "mean": round(float(arr.mean()), 1),
            "is_summary": is_summary,
        }

    text_stats[cat_key] = {
        "label": cat_info["label"],
        "fields": cat_stats,
    }
    for field, stats in cat_stats.items():
        tag = " [요약]" if stats["is_summary"] else ""
        print(f"  {cat_info['label']}.{field}{tag}: P50={stats['p50']:,}, P90={stats['p90']:,}, max={stats['max']:,}")

텍스트 분석:   9%|▉         | 1/11 [00:00<00:02,  3.79it/s]

  판례.판례내용: P50=89, P90=141, max=2,712
  판례.판결요지: P50=369, P90=899, max=12,370
  판례.판시사항: P50=102, P90=283, max=2,712
  판례.이유: P50=2,270, P90=7,562, max=861,517
  판례.판례요약 [요약]: P50=229, P90=320, max=831


텍스트 분석:  36%|███▋      | 4/11 [00:01<00:02,  3.30it/s]

  법령.조문: P50=11,752, P90=48,949, max=747,801
  법령.법령 요약 [요약]: P50=720, P90=1,288, max=2,203
  헌재결정례.판시사항: P50=71, P90=207, max=18,524
  헌재결정례.결정요지: P50=255, P90=740, max=17,145
  헌재결정례.이유: P50=678, P90=8,115, max=297,349
  헌재결정례.심판례요약 [요약]: P50=289, P90=376, max=946
  행정심판례.주문: P50=14, P90=81, max=852
  행정심판례.이유: P50=3,382, P90=9,520, max=93,737
  행정심판례.심판례요약 [요약]: P50=241, P90=306, max=575
  특별행정심판.주문: P50=120, P90=222, max=1,843
  특별행정심판.이유: P50=2,911, P90=7,594, max=88,797
  특별행정심판.청구취지: P50=53, P90=395, max=6,025
  특별행정심판.심판례요약 [요약]: P50=202, P90=251, max=407
  법령해석례.질의요지: P50=394, P90=958, max=3,011
  법령해석례.회답: P50=127, P90=272, max=1,064
  법령해석례.이유: P50=2,557, P90=4,380, max=13,311
  법령해석례.해석례요약 [요약]: P50=200, P90=246, max=380
  위원회 결정문.이유: P50=4,216, P90=7,825, max=16,193
  위원회 결정문.결정요지: P50=106, P90=126, max=272
  위원회 결정문.주문: P50=105, P90=211, max=657
  위원회 결정문.결정문요약 [요약]: P50=271, P90=329, max=433
  부처 해석례.질의요지: P50=70, P90=452, max=2,043
  부처 해석례.회답: P50=439, P90=934, max=3,2

텍스트 분석: 100%|██████████| 11/11 [00:01<00:00,  6.73it/s]

  법률용어사전.법령용어정의: P50=62, P90=286, max=102,778
  조약.조약내용: P50=2,730, P90=14,609, max=400,747
  조약.조약요약 [요약]: P50=232, P90=292, max=473
  행정규칙.조문내용: P50=2,248, P90=7,904, max=64,681
  행정규칙.행정규칙요약 [요약]: P50=532, P90=699, max=1,759


In [6]:
# ── 카테고리별 텍스트 길이 box plot ──────────────────────
# 원본 필드와 요약 필드를 색상으로 구분
box_data = []
for cat_key, ts in text_stats.items():
    for field, stats in ts["fields"].items():
        tag = " [요약]" if stats.get("is_summary") else ""
        box_data.append({
            "카테고리": ts["label"],
            "필드": field,
            "레이블": f"{ts['label']}\n{field}{tag}",
            "P25": stats["p25"],
            "P50": stats["p50"],
            "P75": stats["p75"],
            "P90": stats["p90"],
            "P99": stats["p99"],
            "mean": stats["mean"],
            "유형": "요약" if stats.get("is_summary") else "원본",
        })

df_box = pd.DataFrame(box_data)

# 원본/요약 색상 매핑
color_map = {"원본": "#636EFA", "요약": "#EF553B"}

fig = go.Figure()
for _, row in df_box.iterrows():
    color = color_map[row["유형"]]
    fig.add_trace(go.Box(
        name=row["레이블"],
        q1=[row["P25"]],
        median=[row["P50"]],
        q3=[row["P75"]],
        lowerfence=[row["P25"]],
        upperfence=[row["P90"]],
        mean=[row["mean"]],
        boxmean=True,
        marker_color=color,
        line_color=color,
    ))

fig.update_layout(
    title="카테고리별 텍스트 길이 분포 (P25-P90, 문자 수)<br><sub>🔵 원본 필드  🔴 요약 필드</sub>",
    yaxis_title="텍스트 길이 (문자)",
    height=700,
    showlegend=False,
)
fig.show()

In [7]:
# ── 텍스트 길이 히스토그램 (원본 대표 필드 + 요약 필드) ──
from plotly.subplots import make_subplots

CHUNK_SIZE = 1250  # 현재 청킹 기준선

for cat_key in ["precedent", "law", "constitutional", "administration"]:
    if cat_key not in text_stats or cat_key not in _data_cache:
        continue

    ts = text_stats[cat_key]
    data = _data_cache[cat_key]
    cat_info = CATEGORIES[cat_key]
    summary_field = cat_info.get("summary_field")

    # 원본 대표 필드 (첫 번째 text_field)
    original_fields = [f for f in ts["fields"] if not ts["fields"][f].get("is_summary")]
    if not original_fields:
        continue
    first_field = original_fields[0]

    # 원본 필드 길이 수집
    orig_lengths = [
        len(r.get(first_field, ""))
        for r in data
        if isinstance(r.get(first_field), str) and r.get(first_field, "").strip()
    ]

    # 요약 필드 길이 수집
    summary_lengths = []
    if summary_field:
        summary_lengths = [
            len(r.get(summary_field, ""))
            for r in data
            if isinstance(r.get(summary_field), str) and r.get(summary_field, "").strip()
        ]

    if not orig_lengths:
        continue

    # 요약 필드가 있으면 2열, 없으면 1열
    has_summary = bool(summary_lengths)
    cols = 2 if has_summary else 1

    fig = make_subplots(
        rows=1, cols=cols,
        subplot_titles=[
            f"원본: {first_field}",
            *([ f"요약: {summary_field}"] if has_summary else []),
        ],
        horizontal_spacing=0.1,
    )

    # 원본 필드 히스토그램
    fig.add_trace(
        go.Histogram(x=orig_lengths, nbinsx=100, marker_color="#636EFA", name="원본"),
        row=1, col=1,
    )
    fig.add_vline(
        x=CHUNK_SIZE, line_dash="dash", line_color="red",
        annotation_text=f"청킹 기준 ({CHUNK_SIZE}자)", row=1, col=1,
    )

    # 요약 필드 히스토그램
    if has_summary:
        fig.add_trace(
            go.Histogram(x=summary_lengths, nbinsx=100, marker_color="#EF553B", name="요약"),
            row=1, col=2,
        )
        fig.add_vline(
            x=CHUNK_SIZE, line_dash="dash", line_color="red",
            annotation_text=f"청킹 기준 ({CHUNK_SIZE}자)", row=1, col=2,
        )

    fig.update_layout(
        title=f"{ts['label']} - 텍스트 길이 분포 (원본 vs 요약)",
        height=400,
        showlegend=False,
    )
    fig.update_xaxes(title_text="문자 수")
    fig.update_yaxes(title_text="빈도")
    fig.show()

In [8]:
# ── 예상 청크 수 분석 (원본 + 요약 필드) ─────────────────
# 현재 청킹 설정: 1,250자 / 800토큰 기준
CHUNK_CHAR_LIMIT = 1250
OVERLAP_CHARS = 200  # 가정: 200자 오버랩

chunking_analysis = []
for cat_key, ts in text_stats.items():
    for field, stats in ts["fields"].items():
        # 평균 텍스트 길이 기준 예상 청크 수
        effective_chunk = CHUNK_CHAR_LIMIT - OVERLAP_CHARS
        avg_chunks = max(1, stats["mean"] / effective_chunk) if effective_chunk > 0 else 1
        p90_chunks = max(1, stats["p90"] / effective_chunk) if effective_chunk > 0 else 1

        field_type = "요약" if stats.get("is_summary") else "원본"
        chunking_analysis.append({
            "카테고리": ts["label"],
            "필드": field,
            "유형": field_type,
            "평균 길이": stats["mean"],
            "P90 길이": stats["p90"],
            "평균 청크 수": round(avg_chunks, 1),
            "P90 청크 수": round(p90_chunks, 1),
            "청킹 필요": "예" if stats["p50"] > CHUNK_CHAR_LIMIT else "아니오",
        })

df_chunk = pd.DataFrame(chunking_analysis)

# 유형별 색상 적용
fill_colors = []
for col in df_chunk.columns:
    col_colors = []
    for _, row in df_chunk.iterrows():
        if row["유형"] == "요약":
            col_colors.append("#FFF3F3")  # 연한 빨강
        else:
            col_colors.append("#F2F2F2")
    fill_colors.append(col_colors)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_chunk.columns),
        fill_color="#548235",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[df_chunk[col] for col in df_chunk.columns],
        fill_color=fill_colors,
        align="left",
        font=dict(size=11),
        height=28,
    ),
)])
fig.update_layout(
    title=f"청킹 전략 권고 (기준: {CHUNK_CHAR_LIMIT}자, 오버랩: {OVERLAP_CHARS}자)<br><sub>⬜ 원본 필드  🟥 요약 필드</sub>",
    height=max(400, len(chunking_analysis) * 30 + 100),
)
fig.show()

# ── 요약 필드 vs 원본 필드 청킹 비교 ──────────────────────
print("\n=== 요약 필드 vs 원본 필드 청킹 비교 ===")
for cat_key, ts in text_stats.items():
    orig_fields = {f: s for f, s in ts["fields"].items() if not s.get("is_summary")}
    summary_fields = {f: s for f, s in ts["fields"].items() if s.get("is_summary")}
    if not summary_fields:
        continue

    # 원본 필드 중 가장 긴 것
    if orig_fields:
        longest_orig = max(orig_fields.items(), key=lambda x: x[1]["p50"])
        orig_name, orig_stats = longest_orig
    else:
        continue

    for sum_name, sum_stats in summary_fields.items():
        ratio = sum_stats["p50"] / orig_stats["p50"] if orig_stats["p50"] > 0 else 0
        needs_chunk_orig = "Y" if orig_stats["p50"] > CHUNK_CHAR_LIMIT else "N"
        needs_chunk_sum = "Y" if sum_stats["p50"] > CHUNK_CHAR_LIMIT else "N"
        print(f"  {ts['label']}: {orig_name}(P50={orig_stats['p50']:,}, 청킹={needs_chunk_orig}) → {sum_name}(P50={sum_stats['p50']:,}, 청킹={needs_chunk_sum}) | 요약 비율={ratio:.1%}")


=== 요약 필드 vs 원본 필드 청킹 비교 ===
  판례: 이유(P50=2,270, 청킹=Y) → 판례요약(P50=229, 청킹=N) | 요약 비율=10.1%
  법령: 조문(P50=11,752, 청킹=Y) → 법령 요약(P50=720, 청킹=N) | 요약 비율=6.1%
  헌재결정례: 이유(P50=678, 청킹=N) → 심판례요약(P50=289, 청킹=N) | 요약 비율=42.6%
  행정심판례: 이유(P50=3,382, 청킹=Y) → 심판례요약(P50=241, 청킹=N) | 요약 비율=7.1%
  특별행정심판: 이유(P50=2,911, 청킹=Y) → 심판례요약(P50=202, 청킹=N) | 요약 비율=6.9%
  법령해석례: 이유(P50=2,557, 청킹=Y) → 해석례요약(P50=200, 청킹=N) | 요약 비율=7.8%
  위원회 결정문: 이유(P50=4,216, 청킹=Y) → 결정문요약(P50=271, 청킹=N) | 요약 비율=6.4%
  부처 해석례: 회답(P50=439, 청킹=N) → 해석요약(P50=193, 청킹=N) | 요약 비율=44.0%
  조약: 조약내용(P50=2,730, 청킹=Y) → 조약요약(P50=232, 청킹=N) | 요약 비율=8.5%
  행정규칙: 조문내용(P50=2,248, 청킹=Y) → 행정규칙요약(P50=532, 청킹=N) | 요약 비율=23.7%


In [9]:
# ── 결과 저장 ─────────────────────────────────────────────
p3_path = save_result("phase3_quality", quality_results)
print(f"Phase 3 저장: {p3_path}")

p4_path = save_result("phase4_text", text_stats)
print(f"Phase 4 저장: {p4_path}")

# 요약 출력
print(f"\n=== 품질 요약 ===")
for cat_key, qr in quality_results.items():
    issues = []
    if qr["duplicate_rate"] > 0:
        issues.append(f"ID중복 {qr['duplicate_rate']:.1%}")
    if qr["encoding_issues"] > 0:
        issues.append(f"인코딩 {qr['encoding_issues']}건")
    if qr["html_entities"] > 0:
        issues.append(f"HTML {qr['html_entities']}건")
    issue_str = ", ".join(issues) if issues else "이슈 없음"
    print(f"  {qr['label']}: {issue_str}")

Phase 3 저장: C:\Users\fkjy1\dev\boot_camp\law-3\backend\eda_output\phase3_quality.json
Phase 4 저장: C:\Users\fkjy1\dev\boot_camp\law-3\backend\eda_output\phase4_text.json

=== 품질 요약 ===
  판례: 이슈 없음
  법령: 이슈 없음
  헌재결정례: HTML 3건
  행정심판례: HTML 4건
  특별행정심판: 이슈 없음
  법령해석례: 이슈 없음
  위원회 결정문: 이슈 없음
  부처 해석례: 이슈 없음
  법률용어사전: ID중복 2.2%, HTML 1건
  조약: HTML 6건
  행정규칙: ID중복 0.0%
